In [45]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm.auto import tqdm


In [46]:
torch.manual_seed(44)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

Using cuda


In [47]:
# Transfromation on data
transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307),(0.3081))])

# Load the MNIST dataset
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(mnist_train, batch_size=10, shuffle=True)

mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(mnist_test, batch_size=10, shuffle= True)


In [48]:
class NumClassifier(nn.Module):
  def __init__(self,hidden_size_1=1000,hidden_size_2=2000):
    super().__init__()
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28,hidden_size_1),
        nn.ReLU(),
        nn.Linear(hidden_size_1,hidden_size_2),
        nn.ReLU(),
        nn.Linear(hidden_size_2,10),
    )

  def forward(self,x:torch.Tensor):
      return self.classifier(x)



In [49]:
model = NumClassifier().to(device)
print(model.state_dict().keys())
print(model.parameters)

odict_keys(['classifier.1.weight', 'classifier.1.bias', 'classifier.3.weight', 'classifier.3.bias', 'classifier.5.weight', 'classifier.5.bias'])
<bound method Module.parameters of NumClassifier(
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=1000, bias=True)
    (2): ReLU()
    (3): Linear(in_features=1000, out_features=2000, bias=True)
    (4): ReLU()
    (5): Linear(in_features=2000, out_features=10, bias=True)
  )
)>


In [50]:
def train(train_loader, model , epochs=5,total_iterations_limit=None):


  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
  total_iterations =0

  for epoch in range(epochs):

    model.train()
    loss_sum = 0
    num_iterations = 0

    data_iterator = tqdm(train_loader , desc=f'Epoch {epoch+1}')
    if total_iterations_limit is not None:
      data_iterator.total = total_iterations_limit

    for batch in data_iterator:
      num_iterations += 1
      total_iterations += 1

      X , y = batch
      X = X.to(device)
      y = y.to(device)

      optimizer.zero_grad()

      output = model(X)
      loss = criterion(output,y)

      loss_sum+=loss.item()

      avg_loss = loss_sum/num_iterations

      data_iterator.set_postfix({'loss':f'{avg_loss:.4f}'})

      loss.backward()
      optimizer.step()

      if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
        data_iterator.close()
        break


train(train_loader,model, epochs=1)

Epoch 1:   0%|          | 0/6000 [00:00<?, ?it/s]

In [51]:
x_ = torch.randn(10,1,28,28).to(device)
output = model(x_)
output[0].shape

torch.Size([10])

In [52]:
original_weight = {}

original_weight['classifier.1.weight'] = (
    model.classifier[1].weight.detach().clone()
)

original_weight['classifier.3.weight'] = (
    model.classifier[3].weight.detach().clone()
)

original_weight['classifier.5.weight'] = (
    model.classifier[5].weight.detach().clone()
)

In [53]:
def test():
  correct = 0
  total = 0
  wrong_counts = [0 for i in range(10)]

  model.eval()

  with torch.inference_mode():
    for batch in tqdm(test_loader,desc='Testing'):
      X , y = batch
      X = X.to(device)
      y = y.to(device)

      output = model(X)
      for idx, i in enumerate(output):
        if torch.argmax(i)==y[idx]:
          correct+=1
        else:
          wrong_counts[y[idx]] +=1
        total+=1
    print(f'Accuracy: {round(correct/total, 3)}')
    for i in range(len(wrong_counts)):
        print(f'wrong counts for the digit {i}: {wrong_counts[i]}')

test()


Testing:   0%|          | 0/1000 [00:00<?, ?it/s]

Accuracy: 0.96
wrong counts for the digit 0: 18
wrong counts for the digit 1: 17
wrong counts for the digit 2: 34
wrong counts for the digit 3: 41
wrong counts for the digit 4: 8
wrong counts for the digit 5: 47
wrong counts for the digit 6: 23
wrong counts for the digit 7: 47
wrong counts for the digit 8: 52
wrong counts for the digit 9: 117


In [54]:
model.state_dict().keys()

odict_keys(['classifier.1.weight', 'classifier.1.bias', 'classifier.3.weight', 'classifier.3.bias', 'classifier.5.weight', 'classifier.5.bias'])

In [55]:
model.state_dict().keys()

odict_keys(['classifier.1.weight', 'classifier.1.bias', 'classifier.3.weight', 'classifier.3.bias', 'classifier.5.weight', 'classifier.5.bias'])

In [56]:
total_parameters_original = 0
for index, layer in enumerate(model.state_dict().keys()):
  total_parameters_original += model.state_dict()[layer].numel()
  print(f'Layer {index+1}: {model.state_dict()[layer].numel()}')
print(f'Total Parameters: {total_parameters_original}')


Layer 1: 784000
Layer 2: 1000
Layer 3: 2000000
Layer 4: 2000
Layer 5: 20000
Layer 6: 10
Total Parameters: 2807010


In [57]:
class LoRAParamertrization(nn.Module):

  def __init__(self,feature_in,feature_out,rank=1,alpha=1,device='gpu'):
    super().__init__()

    self.feature_in = feature_in
    self.feature_out = feature_out
    self.rank = rank
    self.alpha = alpha
    self.device = device

    self.lora_A = nn.Parameter(torch.randn(rank,feature_out,device=device))
    self.lora_B = nn.Parameter(torch.randn(feature_in,rank,device=device))
    nn.init.normal_(self.lora_A, mean=0, std=1)

    self.scale = self.alpha/self.rank
    self.enabled = True



  def forward(self,original_weights):
    if self.enabled:
      return original_weights + torch.matmul(self.lora_B,self.lora_A).view(original_weights.shape)*self.scale
    else:
      return original_weights



In [58]:
import torch.nn.utils.parametrize as parametrize

def linear_layer_parameterization(layer,device,rank=1,loara_alpha=1):

  feature_in ,feature_out = layer.weight.shape

  return LoRAParamertrization(feature_in,feature_out,rank,loara_alpha,device)


parametrize.register_parametrization(
    model.classifier[1],
    "weight",
    linear_layer_parameterization(model.classifier[1], device)
)

parametrize.register_parametrization(
    model.classifier[3],
    "weight",
    linear_layer_parameterization(model.classifier[3], device)
)

parametrize.register_parametrization(
    model.classifier[5],
    "weight",
    linear_layer_parameterization(model.classifier[5], device)
)

ParametrizedLinear(
  in_features=2000, out_features=10, bias=True
  (parametrizations): ModuleDict(
    (weight): ParametrizationList(
      (0): LoRAParamertrization()
    )
  )
)

In [59]:
def enable_disable_lora(enabled=True):
    for layer in [model.classifier[1],model.classifier[3], model.classifier[5]]:
        layer.parametrizations["weight"][0].enabled = enabled

In [60]:
total_parameters_lora = 0
total_parameters_non_lora = 0

for index, layer in enumerate([
    model.classifier[1],
    model.classifier[3],
    model.classifier[5]
]):

    lora = layer.parametrizations["weight"][0]

    total_parameters_lora += (
        lora.lora_A.nelement()
        + lora.lora_B.nelement()
    )

    total_parameters_non_lora += (
        layer.weight.nelement()
        + layer.bias.nelement()
    )

    print(
        f'Layer {index+1}: '
        f'W: {layer.weight.shape} + '
        f'B: {layer.bias.shape} + '
        f'LoRA_A: {lora.lora_A.shape} + '
        f'LoRA_B: {lora.lora_B.shape}'
    )

print(f'Total number of parameters (original): {total_parameters_non_lora:,}')
print(f'Total number of parameters (LoRA only): {total_parameters_lora:,}')
print(f'Total number of parameters (original + LoRA): {total_parameters_non_lora + total_parameters_lora:,}')

parameter_increment = (
    total_parameters_lora
    / total_parameters_non_lora
) * 100

print(f'Parameter increase: {parameter_increment:.3f}%')

Layer 1: W: torch.Size([1000, 784]) + B: torch.Size([1000]) + LoRA_A: torch.Size([1, 784]) + LoRA_B: torch.Size([1000, 1])
Layer 2: W: torch.Size([2000, 1000]) + B: torch.Size([2000]) + LoRA_A: torch.Size([1, 1000]) + LoRA_B: torch.Size([2000, 1])
Layer 3: W: torch.Size([10, 2000]) + B: torch.Size([10]) + LoRA_A: torch.Size([1, 2000]) + LoRA_B: torch.Size([10, 1])
Total number of parameters (original): 2,807,010
Total number of parameters (LoRA only): 6,794
Total number of parameters (original + LoRA): 2,813,804
Parameter increase: 0.242%


In [61]:
# Freeze the non-LoRA parameters
for name, param in model.named_parameters():
    # When using torch.nn.utils.parametrize, avoid explicitly setting
    # requires_grad=False on the 'original' weights as it can conflict
    # with parametrization's internal graph management. Biases, however,
    # can still be frozen.
    if 'lora' not in name and 'parametrizations.weight.original' not in name:
        print(f'Freezing non-LoRA parameter {name}')
        param.requires_grad = False

# Load the MNIST dataset again, by keeping only the digit 9
mnist_trainset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

exclude_indices = mnist_trainset.targets == 9

mnist_trainset.data = mnist_trainset.data[exclude_indices]
mnist_trainset.targets = mnist_trainset.targets[exclude_indices]

# Create a dataloader for the training
train_loader = torch.utils.data.DataLoader(
    mnist_trainset,
    batch_size=10,
    shuffle=True
)

# Train the network with LoRA only on the digit 9
# and only for 100 batches
train(
    train_loader,
    model,
    epochs=1,
    total_iterations_limit=100
)


Freezing non-LoRA parameter classifier.1.bias
Freezing non-LoRA parameter classifier.3.bias
Freezing non-LoRA parameter classifier.5.bias


Epoch 1:   0%|          | 0/595 [00:00<?, ?it/s]

In [62]:
# Test with LoRA enabled
enable_disable_lora(enabled=True)
test()

Testing:   0%|          | 0/1000 [00:00<?, ?it/s]

Accuracy: 0.111
wrong counts for the digit 0: 974
wrong counts for the digit 1: 1134
wrong counts for the digit 2: 1031
wrong counts for the digit 3: 977
wrong counts for the digit 4: 977
wrong counts for the digit 5: 892
wrong counts for the digit 6: 876
wrong counts for the digit 7: 1027
wrong counts for the digit 8: 973
wrong counts for the digit 9: 28


In [63]:
# Test with LoRA disabled
enable_disable_lora(enabled=False)
test()

Testing:   0%|          | 0/1000 [00:00<?, ?it/s]

Accuracy: 0.879
wrong counts for the digit 0: 180
wrong counts for the digit 1: 13
wrong counts for the digit 2: 41
wrong counts for the digit 3: 97
wrong counts for the digit 4: 382
wrong counts for the digit 5: 121
wrong counts for the digit 6: 39
wrong counts for the digit 7: 229
wrong counts for the digit 8: 97
wrong counts for the digit 9: 8
